# Mục 3b(ii) — Điều chỉnh 10% các giá trị platelets cực trị
**Võ Minh Huy — 2311110117**

Thực hiện cùng yêu cầu với SQL 05c: tăng 10% cho 5 giá trị nhỏ nhất, giảm 10% cho 5 giá trị lớn nhất của thuộc tính có độ lệch chuẩn lớn nhất. Dùng đầu vào sau N07, trước làm mịn nhóm; lưu kết quả riêng.

## 1. Đọc đầu vào và chọn 10 hồ sơ
Chọn từ giá trị trước cập nhật. Nếu bằng nhau, dùng MaHoSo để xác định thứ tự trong cùng bản làm việc. MaHoSo có thể khác SQL.

In [1]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_vao = thu_muc / "ket_qua/11_dien_ejection_fraction/ban_lam_viec_sau_ef_python.csv"
ban_dau = pd.read_csv(file_vao, dtype=str, keep_default_na=False)
ban_dau["MaHoSo"] = ban_dau["MaHoSo"].astype(int)
ban_dau["platelets"] = ban_dau["platelets"].map(Decimal)
if len(ban_dau) < 10 or ban_dau["platelets"].isna().any():
    raise ValueError("Cần ít nhất 10 hồ sơ và platelets không còn thiếu.")

tang_dan = ban_dau.sort_values(["platelets", "MaHoSo"])
giam_dan = ban_dau.sort_values(["platelets", "MaHoSo"], ascending=False)
nhom_nho = tang_dan.head(5)
nhom_lon = giam_dan.head(5)
pd.DataFrame({"SoHoSo":[len(ban_dau)], "SoChonNho":[len(nhom_nho)],
              "SoChonLon":[len(nhom_lon)]})

,SoHoSo,SoChonNho,SoChonLon
0,299,5,5


## 2. Điều chỉnh và tổng kết
Giá trị nhỏ × 1,10; giá trị lớn × 0,90. Dùng Decimal và ROUND_HALF_UP đến 2 chữ số, tương ứng ROUND trong SQL. Chỉ cập nhật platelets trên bản sao.

In [2]:
ban_dieu_chinh = ban_dau.copy(deep=True)
dong_nhat_ky = []
for ten_nhom, nhom, ty_le in [("5 nhỏ nhất", nhom_nho, 10),
                              ("5 lớn nhất", nhom_lon, -10)]:
    he_so = Decimal("1.10") if ty_le == 10 else Decimal("0.90")
    for hang, (i, r) in enumerate(nhom.iterrows(), start=1):
        truoc = r["platelets"]
        sau = (truoc * he_so).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        ban_dieu_chinh.at[i, "platelets"] = sau
        dong_nhat_ky.append({"Nhom":ten_nhom, "ThuHang":hang,
            "MaHoSo":r["MaHoSo"], "PlateletsTruoc":truoc,
            "TyLePhanTram":ty_le, "PlateletsSau":sau, "ChenhLech":sau-truoc})
nhat_ky = pd.DataFrame(dong_nhat_ky)
tong_ket = pd.DataFrame({"SoHoSo":[len(ban_dieu_chinh)],
    "SoTang10PhanTram":[nhat_ky["TyLePhanTram"].eq(10).sum()],
    "SoGiam10PhanTram":[nhat_ky["TyLePhanTram"].eq(-10).sum()],
    "SoGiaTriThayDoi":[ban_dieu_chinh["platelets"].ne(ban_dau["platelets"]).sum()],
    "SoHoSoGiuNguyen":[ban_dieu_chinh["platelets"].eq(ban_dau["platelets"]).sum()]})
tong_ket

,SoHoSo,SoTang10PhanTram,SoGiam10PhanTram,SoGiaTriThayDoi,SoHoSoGiuNguyen
0,299,5,5,10,289


## 3. Nhật ký đủ 10 hồ sơ trước sau
Đối chiếu SQL bằng nhóm, thứ hạng, giá trị trước/sau, tỷ lệ và chênh lệch; không dùng MaHoSo giữa hai môi trường.

In [3]:
nhat_ky

,Nhom,ThuHang,MaHoSo,PlateletsTruoc,TyLePhanTram,PlateletsSau,ChenhLech
0,5 nhỏ nhất,1,278,25100.000000,10,27610.00,2510.000000
1,5 nhỏ nhất,2,16,47000.000000,10,51700.00,4700.000000
2,5 nhỏ nhất,3,282,51000.000000,10,56100.00,5100.000000
3,5 nhỏ nhất,4,231,62000.000000,10,68200.00,6200.000000
4,5 nhỏ nhất,5,168,70000.000000,10,77000.00,7000.000000
5,5 lớn nhất,1,110,850000.000000,-10,765000.00,-85000.000000
6,5 lớn nhất,2,297,742000.000000,-10,667800.00,-74200.000000
7,5 lớn nhất,3,106,621000.000000,-10,558900.00,-62100.000000
8,5 lớn nhất,4,288,543000.000000,-10,488700.00,-54300.000000
9,5 lớn nhất,5,118,507000.000000,-10,456300.00,-50700.000000


**Nhận xét:** 299 hồ sơ được giữ lại; đúng 5 giá trị tăng, 5 giá trị giảm, 289 hồ sơ không đổi. Các cột khác giữ nguyên. Phép này được thực hiện riêng với phép thay mean của 5 nhóm.

## 4. Lưu kết quả
Lưu tổng kết, nhật ký và bản làm việc trung gian; giữ nguyên CSV đầu vào.

In [4]:
thu_muc_kq = thu_muc / "ket_qua/13_lam_min/03_dieu_chinh_10_phan_tram"
thu_muc_kq.mkdir(parents=True, exist_ok=True)
tong_ket.to_csv(thu_muc_kq / "tong_ket_python.csv", index=False, encoding="utf-8-sig")
nhat_ky.to_csv(thu_muc_kq / "nhat_ky_python.csv", index=False, encoding="utf-8-sig")
ban_dieu_chinh.to_csv(thu_muc_kq / "ban_lam_viec_sau_10_phan_tram_python.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"TrangThai":["Đã lưu kết quả điều chỉnh 10%; giữ nguyên đầu vào."]})

,TrangThai
0,Đã lưu kết quả điều chỉnh 10%; giữ nguyên đầu ...
